In [1]:
%load_ext autoreload
%autoreload 2

import os
import concurrent.futures
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import multiprocessing as mp


try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass

from attractor import safe_run_single_simulation, safe_run_single_simulation_with_functional_rf
from visualization import plot_diff_matrix
from mean_and_entropy import calculate_mean, calculate_entropy
from attractor_analysis import (
    plot_attractor_activity,
    analyze_averaged_attractor_trajectory,
    plot_attractor_energy_landscape,
    get_full_rate_matrix,
    calculate_lyapunov_exponent,
    merge_functional_receptive_field_data,
    analyze_functional_receptive_field,
    compare_functional_receptive_fields,
)


              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.8.0-post0.dev0
 Built: Feb 13 2025 14:17:07

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.



In [2]:
RATES_TO_RUN = [40, 50, 60]
RATE_VAL = RATES_TO_RUN[0]
N_TRIALS = 20
NUM_EXC_NEURONS = 10

RUN_FUNCTIONAL_RF = True
RF_REPEATS = 3
RF_STIMULUS_TIME = 500.0
RF_QUIET_TIME = 200.0
RF_STIMULUS_CENTERS = [25 + 50 * k for k in range(NUM_EXC_NEURONS)]

In [3]:
MAX_WORKERS = max(1, os.cpu_count() - 4)
print(f"{os.cpu_count()} CPU cores detected; running {MAX_WORKERS} parallel processes.")

20 CPU cores detected; running 16 parallel processes.


In [ ]:
# Batch simulation for rates 40, 50, and 60.
init_w_matrix = np.ones((NUM_EXC_NEURONS, NUM_EXC_NEURONS)) * 0.1
init_w_matrix -= np.identity(NUM_EXC_NEURONS) * 0.1

batch_results = {}
weight_encodings = []
weight_consolidations = []
entropy_encodings = []
entropy_consolidations = []
mean_encodings = []
mean_consolidations = []

for RATE_VAL in RATES_TO_RUN:
    print(f"\n====== Rate {RATE_VAL}: Post-Training Attractor Test ({N_TRIALS} Trials in Parallel) ======")
    w_enc_train_all = []
    spikes_early_all = []
    rf_train_data_all = []

    with concurrent.futures.ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        if RUN_FUNCTIONAL_RF:
            futures = [
                executor.submit(
                    safe_run_single_simulation_with_functional_rf,
                    RATE_VAL,
                    True,
                    RF_STIMULUS_CENTERS,
                    RF_REPEATS,
                    RF_STIMULUS_TIME,
                    RF_QUIET_TIME,
                )
                for _ in range(N_TRIALS)
            ]
        else:
            futures = [executor.submit(safe_run_single_simulation, RATE_VAL, True) for _ in range(N_TRIALS)]

        for future in tqdm(concurrent.futures.as_completed(futures), total=N_TRIALS, desc=f"Rate {RATE_VAL} Early Trials"):
            result = future.result()
            if RUN_FUNCTIONAL_RF:
                w_enc, _, spikes_early, rf_train_data = result
                rf_train_data_all.append(rf_train_data)
            else:
                w_enc, _, spikes_early = result
            w_enc_train_all.append(w_enc)
            spikes_early_all.append(spikes_early)

    avg_w_enc = np.mean(w_enc_train_all, axis=0)
    weight_encodings.append(avg_w_enc)
    entropy_encodings.append(calculate_entropy(avg_w_enc))
    mean_encodings.append(calculate_mean(avg_w_enc))

    plot_attractor_activity(spikes_early_all[0], RATE_VAL, phase_name="Training", trial_idx=1, num_neurons=NUM_EXC_NEURONS)
    pca_traj_early = analyze_averaged_attractor_trajectory(
        spikes_early_all,
        rate_val=RATE_VAL,
        phase_name="Training",
        num_neurons=NUM_EXC_NEURONS,
        color='red',
    )
    plot_attractor_energy_landscape(pca_traj_early, phase_name="Training", rate_val=RATE_VAL)
    rate_matrix_train = get_full_rate_matrix(spikes_early_all, num_neurons=NUM_EXC_NEURONS)

    rf_train_result = None
    if RUN_FUNCTIONAL_RF:
        rf_train_data = merge_functional_receptive_field_data(rf_train_data_all)
        rf_train_result = analyze_functional_receptive_field(
            rf_train_data,
            rate_val=RATE_VAL,
            phase_name="PostTraining",
            num_neurons=NUM_EXC_NEURONS,
        )

    print(f"\n====== Rate {RATE_VAL}: Post-Consolidation Attractor Test ({N_TRIALS} Trials in Parallel) ======")
    w_cons_all = []
    spikes_late_all = []
    rf_cons_data_all = []

    with concurrent.futures.ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        if RUN_FUNCTIONAL_RF:
            futures = [
                executor.submit(
                    safe_run_single_simulation_with_functional_rf,
                    RATE_VAL,
                    False,
                    RF_STIMULUS_CENTERS,
                    RF_REPEATS,
                    RF_STIMULUS_TIME,
                    RF_QUIET_TIME,
                )
                for _ in range(N_TRIALS)
            ]
        else:
            futures = [executor.submit(safe_run_single_simulation, RATE_VAL, False) for _ in range(N_TRIALS)]

        for future in tqdm(concurrent.futures.as_completed(futures), total=N_TRIALS, desc=f"Rate {RATE_VAL} Late Trials"):
            result = future.result()
            if RUN_FUNCTIONAL_RF:
                _, w_cons, spikes_late, rf_cons_data = result
                rf_cons_data_all.append(rf_cons_data)
            else:
                _, w_cons, spikes_late = result
            w_cons_all.append(w_cons)
            spikes_late_all.append(spikes_late)

    avg_w_cons = np.mean(w_cons_all, axis=0)
    weight_consolidations.append(avg_w_cons)
    entropy_consolidations.append(calculate_entropy(avg_w_cons))
    mean_consolidations.append(calculate_mean(avg_w_cons))

    plot_attractor_activity(spikes_late_all[0], RATE_VAL, phase_name="Consolidation", trial_idx=1, num_neurons=NUM_EXC_NEURONS)
    pca_traj_late = analyze_averaged_attractor_trajectory(
        spikes_late_all,
        rate_val=RATE_VAL,
        phase_name="Consolidation",
        num_neurons=NUM_EXC_NEURONS,
        color='blue',
    )
    plot_attractor_energy_landscape(pca_traj_late, phase_name="Consolidation", rate_val=RATE_VAL)
    rate_matrix_cons = get_full_rate_matrix(spikes_late_all, num_neurons=NUM_EXC_NEURONS)

    rf_cons_result = None
    rf_change_result = None
    if RUN_FUNCTIONAL_RF:
        rf_cons_data = merge_functional_receptive_field_data(rf_cons_data_all)
        rf_cons_result = analyze_functional_receptive_field(
            rf_cons_data,
            rate_val=RATE_VAL,
            phase_name="PostConsolidation",
            num_neurons=NUM_EXC_NEURONS,
        )
        rf_change_result = compare_functional_receptive_fields(rf_train_result, rf_cons_result, rate_val=RATE_VAL)

    lyapunov_result = calculate_lyapunov_exponent(rate_matrix_train, rate_matrix_cons, RATE_VAL)

    batch_results[RATE_VAL] = {
        "spikes_training": spikes_early_all,
        "spikes_consolidation": spikes_late_all,
        "pca_training": pca_traj_early,
        "pca_consolidation": pca_traj_late,
        "weight_training": avg_w_enc,
        "weight_consolidation": avg_w_cons,
        "rate_matrix_training": rate_matrix_train,
        "rate_matrix_consolidation": rate_matrix_cons,
        "rf_training": rf_train_result,
        "rf_consolidation": rf_cons_result,
        "rf_change": rf_change_result,
        "lyapunov": lyapunov_result,
    }

fig, axes = plt.subplots(len(RATES_TO_RUN), 2, figsize=(15, 5 * len(RATES_TO_RUN)))
if len(RATES_TO_RUN) == 1:
    axes = np.asarray([axes])

for row_idx, r_val in enumerate(RATES_TO_RUN):
    plot_diff_matrix(
        axes[row_idx, 0],
        weight_encodings[row_idx],
        init_w_matrix,
        title_str=f"Train-Init (rate={r_val})",
    )
    plot_diff_matrix(
        axes[row_idx, 1],
        weight_consolidations[row_idx],
        init_w_matrix,
        title_str=f"Consolidation-Init (rate={r_val})",
    )

plt.tight_layout()
os.makedirs("results/figures", exist_ok=True)
plt.savefig("results/figures/Exc_EE_Weight_Matrix_Diff_Rates_40_50_60.pdf", format="pdf", bbox_inches="tight", dpi=300)
plt.show()

plt.figure(figsize=(10, 6), facecolor="white")
plt.plot(RATES_TO_RUN, entropy_encodings, marker="o", label="After Training Entropy", color="blue")
plt.plot(RATES_TO_RUN, entropy_consolidations, marker="s", label="After Consolidation Entropy", color="orange")
plt.title("Entropy of Excitatory Weight Matrices")
plt.xlabel("Spike Rate")
plt.ylabel("Entropy")
plt.xticks(RATES_TO_RUN)
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/Exc_EE_Weight_Entropy_Rates_40_50_60.pdf", format="pdf", bbox_inches="tight", dpi=300)
plt.show()

plt.figure(figsize=(10, 6), facecolor="white")
plt.plot(RATES_TO_RUN, mean_encodings, marker="o", label="After Training Mean", color="blue")
plt.plot(RATES_TO_RUN, mean_consolidations, marker="s", label="After Consolidation Mean", color="orange")
plt.title("Mean of Excitatory Weight Matrices")
plt.xlabel("Spike Rate")
plt.ylabel("Mean")
plt.xticks(RATES_TO_RUN)
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/Exc_EE_Weight_Mean_Rates_40_50_60.pdf", format="pdf", bbox_inches="tight", dpi=300)
plt.show()

print("\nAll requested rates finished:", RATES_TO_RUN)

====== Running Post-Training Attractor Test (20 Trials in Parallel) ======


Early Attractor Trials:   0%|          | 0/20 [00:00<?, ?it/s]


              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.8.0-post0.dev0
 Built: Feb 13 2025 14:17:07

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.


              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.8.0-post0.dev0
 Built: Feb 13 2025 14:17:07

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.


              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.8.0-post0.dev0
 Built: Feb 13 2025 14:17:07

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find

Early Attractor Trials:   5%|▌         | 1/20 [08:05<2:33:40, 485.26s/it]


Jun 03 17:12:07 SimulationManager::run [Info]: 
    Simulation finished.

Jun 03 17:12:08 SimulationManager::set_status [Info]: 
    Temporal resolution changed from 0.1 to 0.05 ms.

Jun 03 17:12:08 NodeManager::prepare_nodes [Info]: 
    Preparing 1018 nodes for simulation.

Jun 03 17:12:08 SimulationManager::start_updating_ [Info]: 
    Number of local nodes: 1018
    Simulation time (ms): 500
    Number of OpenMP threads: 1
    Not using MPI


...


Jun 03 17:12:59 SimulationManager::run [Info]: 
    Simulation finished.

Jun 03 17:12:59 NodeManager::prepare_nodes [Info]: 
    Preparing 1018 nodes for simulation.

Jun 03 17:12:59 SimulationManager::start_updating_ [Info]: 
    Number of local nodes: 1018
    Simulation time (ms): 100
    Number of OpenMP threads: 1
    Not using MPI

Jun 03 17:13:00 SimulationManager::run [Info]: 
    Simulation finished.

Jun 03 17:13:00 SimulationManager::run [Info]: 
    Simulation finished.

Jun 03 17:13:00 NodeManager::prepare_nodes [Info]: 
    Preparing 1018 nodes for simulation.

Jun 03 17:13:00 SimulationManager::start_updating_ [Info]: 
    Number of local nodes: 1018
    Simulation time (ms): 100
    Number of OpenMP threads: 1
    Not using MPI

Jun 03 17:13:00 NodeManager::prepare_nodes [Info]: 
    Preparing 1018 nodes for simulation.

Jun 03 17:13:00 SimulationManager::start_updating_ [Info]: 
    Number of local nodes: 1018
    Simulation time (ms): 100
    Number of OpenMP thread

Early Attractor Trials:  75%|███████▌  | 15/20 [08:57<00:23,  4.65s/it]


Jun 03 17:13:00 SimulationManager::run [Info]: 
    Simulation finished.

Jun 03 17:13:00 NodeManager::prepare_nodes [Info]: 
    Preparing 1018 nodes for simulation.

Jun 03 17:13:00 SimulationManager::start_updating_ [Info]: 
    Number of local nodes: 1018
    Simulation time (ms): 100
    Number of OpenMP threads: 1
    Not using MPI

Jun 03 17:13:00 SimulationManager::run [Info]: 
    Simulation finished.

Jun 03 17:13:00 SimulationManager::run [Info]: 
    Simulation finished.

Jun 03 17:13:00 NodeManager::prepare_nodes [Info]: 
    Preparing 1018 nodes for simulation.

Jun 03 17:13:00 SimulationManager::start_updating_ [Info]: 
    Number of local nodes: 1018
    Simulation time (ms): 100
    Number of OpenMP threads: 1
    Not using MPI

Jun 03 17:13:00 NodeManager::prepare_nodes [Info]: 
    Preparing 1018 nodes for simulation.

Jun 03 17:13:00 SimulationManager::start_updating_ [Info]: 
    Number of local nodes: 1018
    Simulation time (ms): 100
    Number of OpenMP thread

Early Attractor Trials:  80%|████████  | 16/20 [08:58<00:14,  3.59s/it]


Jun 03 17:13:01 NodeManager::prepare_nodes [Info]: 
    Preparing 1018 nodes for simulation.

Jun 03 17:13:01 SimulationManager::start_updating_ [Info]: 
    Number of local nodes: 1018
    Simulation time (ms): 100
    Number of OpenMP threads: 1
    Not using MPI

Jun 03 17:13:01 SimulationManager::run [Info]: 
    Simulation finished.

Jun 03 17:13:01 SimulationManager::run [Info]: 
    Simulation finished.

Jun 03 17:13:01 NodeManager::prepare_nodes [Info]: 
    Preparing 1018 nodes for simulation.

Jun 03 17:13:01 SimulationManager::start_updating_ [Info]: 
    Number of local nodes: 1018
    Simulation time (ms): 100
    Number of OpenMP threads: 1
    Not using MPI

Jun 03 17:13:01 NodeManager::prepare_nodes [Info]: 
    Preparing 1018 nodes for simulation.

Jun 03 17:13:01 SimulationManager::start_updating_ [Info]: 
    Number of local nodes: 1018
    Simulation time (ms): 100
    Number of OpenMP threads: 1
    Not using MPI

Jun 03 17:13:01 SimulationManager::run [Info]: 
  

In [ ]:
# The multi-rate batch loop above already runs post-consolidation for every rate.

In [ ]:
# The multi-rate batch loop above already runs the Lyapunov analysis for every rate.